# 04 — Evaluation & Interpretability

Goals:
- Full metric report on the held-out test set
- PR curve, ROC curve, calibration curve, confusion matrix
- SHAP global summary (beeswarm + bar)
- Partial dependence plots for top 3 features
- SHAP leakage sanity check (no single feature > 40% importance)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
import json

from src.model.train import get_feature_cols
from src.model.evaluate import (
    evaluate_model, plot_shap_summary,
    get_top_shap_factors, plot_partial_dependence
)

ARTIFACTS_DIR = Path('../backend/artifacts')
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Ready.')

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────
model_files = sorted(ARTIFACTS_DIR.glob('model_*.txt'), key=lambda p: p.stat().st_mtime, reverse=True)
if not model_files:
    raise FileNotFoundError(f'No model file found in {ARTIFACTS_DIR}. Run notebook 03 first.')

model_path = model_files[0]
model = lgb.Booster(model_file=str(model_path))
print(f'Loaded: {model_path.name}')

# Load threshold from params.json
params_path = ARTIFACTS_DIR / 'params.json'
threshold = 0.5
if params_path.exists():
    with open(params_path) as f:
        params = json.load(f)
    threshold = params.get('threshold', 0.5)
print(f'Decision threshold: {threshold}')

In [ ]:
# ── Load test features ─────────────────────────────────────────────────────
test_df = pd.read_parquet('../data/test_features.parquet')
feat_cols = get_feature_cols(test_df)
X_test = test_df[feat_cols]
y_test = test_df['y']
print(f'Test set: {X_test.shape} | Delay rate: {y_test.mean():.3f}')

In [ ]:
# ── Full evaluation ────────────────────────────────────────────────────────
metrics = evaluate_model(
    model, X_test, y_test,
    threshold=threshold,
    output_dir=REPORTS_DIR,
)

print('\n═══ TEST SET RESULTS ═══')
print(f'  PR-AUC (primary):         {metrics["pr_auc"]:.4f}')
print(f'  ROC-AUC:                  {metrics["roc_auc"]:.4f}')
print(f'  F1 @ threshold={threshold:.2f}:    {metrics["f1"]:.4f}')
print(f'  Precision @ 50% Recall:   {metrics["precision_at_50_recall"]:.4f}')
print(f'  Recall @ 80% Precision:   {metrics["recall_at_80_precision"]:.4f}')
print(f'  Brier Score:              {metrics["brier_score"]:.4f}')
print(f'  Optimal Threshold (F1):   {metrics["optimal_threshold"]:.4f}')
print()
print(metrics['classification_report'])

In [ ]:
# ── SHAP global summary ────────────────────────────────────────────────────
# Use a sample of 2000 rows for speed
X_shap = X_test.sample(min(2000, len(X_test)), random_state=42)
shap_values = plot_shap_summary(model, X_shap, output_dir=REPORTS_DIR)

In [ ]:
# ── SHAP leakage sanity check ──────────────────────────────────────────────
mean_abs_shap = np.abs(shap_values).mean(axis=0)
total_shap    = mean_abs_shap.sum()
shap_importance_pct = 100 * mean_abs_shap / total_shap

shap_df = pd.DataFrame({
    'feature':    X_shap.columns,
    'mean_abs':   mean_abs_shap,
    'pct':        shap_importance_pct,
}).sort_values('pct', ascending=False)

print('\nSHAP importance (% of total):')
print(shap_df[['feature','pct']].to_string(index=False))

top_feat = shap_df.iloc[0]
if top_feat['pct'] > 40:
    print(f'\n⚠️  WARNING: Feature "{top_feat["feature"]}" has {top_feat["pct"]:.1f}% '
          'SHAP importance — investigate for potential leakage!')
else:
    print(f'\n✅ Leakage sanity check passed. Top feature: {top_feat["feature"]} ({top_feat["pct"]:.1f}%)')

In [ ]:
# ── Partial dependence plots (top 3 features) ──────────────────────────────
plot_partial_dependence(model, X_shap, top_n=3, output_dir=REPORTS_DIR)
print('PDPs saved to reports/')

In [ ]:
# ── Example single-row prediction with SHAP factors ────────────────────────
row = X_test.iloc[[0]]
prob = float(model.predict(row)[0])
factors = get_top_shap_factors(model, row, n=5)

print(f'Delay probability: {prob:.4f}')
print('Top 5 SHAP factors:')
for f in factors:
    direction = '↑ delays' if f['impact'] > 0 else '↓ on-time'
    print(f"  {f['feature']:<30} {f['impact']:+.4f}  {direction}")